Dataset Link- https://www.kaggle.com/datasets/gauravtopre/credit-card-defaulter-prediction

In [5]:
!pip install pandas scikit-learn openpyxl

In [6]:
!ls /content/

CodeAlpha_Credit_Scoring_Model	sample_data


In [11]:
!find /content/CodeAlpha_Credit_Scoring_Model -print

/content/CodeAlpha_Credit_Scoring_Model
/content/CodeAlpha_Credit_Scoring_Model/.git
/content/CodeAlpha_Credit_Scoring_Model/.git/branches
/content/CodeAlpha_Credit_Scoring_Model/.git/info
/content/CodeAlpha_Credit_Scoring_Model/.git/info/exclude
/content/CodeAlpha_Credit_Scoring_Model/.git/description
/content/CodeAlpha_Credit_Scoring_Model/.git/refs
/content/CodeAlpha_Credit_Scoring_Model/.git/refs/heads
/content/CodeAlpha_Credit_Scoring_Model/.git/refs/heads/main
/content/CodeAlpha_Credit_Scoring_Model/.git/refs/tags
/content/CodeAlpha_Credit_Scoring_Model/.git/refs/remotes
/content/CodeAlpha_Credit_Scoring_Model/.git/refs/remotes/origin
/content/CodeAlpha_Credit_Scoring_Model/.git/refs/remotes/origin/HEAD
/content/CodeAlpha_Credit_Scoring_Model/.git/objects
/content/CodeAlpha_Credit_Scoring_Model/.git/objects/pack
/content/CodeAlpha_Credit_Scoring_Model/.git/objects/pack/pack-c2b88dbec6c17b8a099607be98c509b696163b3f.idx
/content/CodeAlpha_Credit_Scoring_Model/.git/objects/pack/pack

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/CodeAlpha_ML_Tasks/Task-1/Credit Card Defaulter Prediction.csv')

print(data.head())

   ID  LIMIT_BAL SEX   EDUCATION MARRIAGE  AGE  PAY_0  PAY_2  PAY_3  PAY_4  \
0   1      20000   F  University  Married   24      2      2     -1     -1   
1   2     120000   F  University   Single   26     -1      2      0      0   
2   3      90000   F  University   Single   34      0      0      0      0   
3   4      50000   F  University  Married   37      0      0      0      0   
4   5      50000   M  University  Married   57     -1      0     -1      0   

   ...  BILL_AMT4  BILL_AMT5  BILL_AMT6  PAY_AMT1  PAY_AMT2  PAY_AMT3  \
0  ...          0          0          0         0       689         0   
1  ...       3272       3455       3261         0      1000      1000   
2  ...      14331      14948      15549      1518      1500      1000   
3  ...      28314      28959      29547      2000      2019      1200   
4  ...      20940      19146      19131      2000     36681     10000   

   PAY_AMT4  PAY_AMT5  PAY_AMT6  default   
0         0         0         0         Y  
1   

In [17]:
%cd /content/CodeAlpha_Credit_Scoring_Model


/content/CodeAlpha_Credit_Scoring_Model


Preprocess The Data

In [32]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = pd.read_csv('/content/drive/MyDrive/CodeAlpha_ML_Tasks/Task-1/Credit Card Defaulter Prediction.csv')

cleaned_columns = []
for col in data.columns:
    cleaned_col = re.sub(r'\\s+', '', col).lower()
    cleaned_columns.append(cleaned_col)
data.columns = cleaned_columns

print("DataFrame columns AFTER initial cleaning:", data.columns.tolist())

categorical_cols = ['sex', 'education', 'marriage']


data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)

print("DataFrame columns AFTER one-hot encoding:", data.columns.tolist())

data['default '] = data['default '].astype(str).str.strip().str.upper()

print("Value counts of 'default' column BEFORE final mapping:\n", data['default '].value_counts(dropna=False))

data['default '] = data['default '].map({'Y': 1, 'N': 0})

print("Value counts of 'default' column AFTER final mapping:\n", data['default '].value_counts(dropna=False))

data.dropna(inplace=True)

print(f"DataFrame shape after dropping NaNs: {data.shape}")

if 'default ' not in data.columns:
    raise ValueError("The 'default ' column is missing after preprocessing.")

X = data.drop('default ', axis=1)
y = data['default ']

if y.empty:
    raise ValueError("The target variable 'y' is empty after preprocessing. \n" \
                     "This means all 'default' values were either missing or not 'Y'/'N' and were dropped.")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


DataFrame columns AFTER initial cleaning: ['id', 'limit_bal', 'sex', 'education', 'marriage', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6', 'bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6', 'default ']
DataFrame columns AFTER one-hot encoding: ['id', 'limit_bal', 'age', 'pay_0', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6', 'bill_amt1', 'bill_amt2', 'bill_amt3', 'bill_amt4', 'bill_amt5', 'bill_amt6', 'pay_amt1', 'pay_amt2', 'pay_amt3', 'pay_amt4', 'pay_amt5', 'pay_amt6', 'default ', 'sex_M', 'education_Graduate school', 'education_High School', 'education_Others', 'education_University', 'education_Unknown', 'marriage_Married', 'marriage_Other', 'marriage_Single']
Value counts of 'default' column BEFORE final mapping:
 default 
N    23364
Y     6636
Name: count, dtype: int64
Value counts of 'default' column AFTER final mapping:
 default 
0    23364
1     6636
Name: count, d

Build and Tarin



In [33]:
from sklearn.ensemble import RandomForestClassifier


model = RandomForestClassifier(random_state=42)


model.fit(X_train, y_train)

y_pred = model.predict(X_test)

Evalute the Model

In [34]:
from sklearn.metrics import classification_report, roc_auc_score

# Model Evaluation
print("Classification Report:\n", classification_report(y_test, y_pred))
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_pred)}')

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.94      0.89      4687
           1       0.63      0.35      0.45      1313

    accuracy                           0.81      6000
   macro avg       0.73      0.65      0.67      6000
weighted avg       0.79      0.81      0.79      6000

ROC-AUC Score: 0.6472973568056449
